# Data 207 Modeling - Nate del Rosario

This notebook is solely for modeling. All EDA and necessary plots can be found in **'src/eda.ipynb'**

In [16]:
import pandas as pd
import numpy as np
import seaborn as sns
import xgboost as xgb
from matplotlib import pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

In [17]:
codebook = pd.read_csv('../data/raw/2017_2018/codebook_2017_2018.csv')
data = pd.read_parquet('../data/model_ready_boostrap_imputed.parquet')
data.head()

,SEQN,age,2_year_interview_weight,2_year_mec_exam_weight,heart_rate,diastolic_bp_1,diastolic_bp_2,diastolic_bp_3,mil,systolic_bp_1,...,dpq_3,dpq_4,dpq_5,dpq_6,dpq_7,dpq_8,dpq_9,dpq_total,depression_category,depression_binary
0,93705.0,1.241848,-0.630069,-0.607551,-1.544887e-15,-1.052688e-15,-1.044279e-15,-0.459920,4.152050,8.620112e-16,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Minimal or no depression,False
1,93706.0,-0.640586,-0.631663,-0.598668,-1.544887e-15,4.560716e-01,1.250459e-01,0.564596,-0.393858,-5.660062e-01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Minimal or no depression,False
2,93708.0,1.241848,-0.516057,-0.468330,-1.544887e-15,-1.052688e-15,7.129227e-01,0.564596,1.310857,8.620112e-16,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Minimal or no depression,False
3,93711.0,0.849674,-0.568076,-0.514050,-1.544887e-15,1.161337e-02,-2.192334e-02,-0.313560,-0.962097,-8.086407e-01,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,Minimal or no depression,False
4,93712.0,-0.640586,-0.136145,-0.099997,-1.544887e-15,1.161337e-02,2.720151e-01,0.125518,-0.393858,-5.660062e-01,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Minimal or no depression,False


In [18]:
print(data.shape)

(5068, 163)


In [19]:
# numerical columns
scale_cols = codebook[codebook['Numeric'] == True]['Column Alias'].values

# encoding target variable
ordinal_map = {
    "Minimal or no depression": 0,
    "Mild depression": 1,
    "Moderate depression": 2,
    "Moderately severe depression": 3,
    "Severe depression": 4
}

# make sure to restart kernel if running this cell again
data["depression_category"] = data["depression_category"].map(ordinal_map)
data["depression_category"].value_counts()

depression_category
0    3772
1     837
2     292
3     124
4      43
Name: count, dtype: int64

In [20]:
X = data[scale_cols]
y = data["depression_category"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Handling Class Imbalance

As a baseline, we stick to class weighting as opposed to upsampling mainly for simplicity and room to expand upon.

In [ ]:
classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

weight_dict = dict(zip(classes, class_weights))
sample_weights = y_train.map(weight_dict)

### Feature Selection on Numeric Columns Only

### Feature Importance via XGBoost

We use XGBoost to see how valuable each feature is in constructing its decision trees

In [22]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train, sample_weight=sample_weights)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=300, n_jobs=-1, num_parallel_tree=None,
              objective='multi:softprob', predictor=None, ...)

In [23]:
# selection based on importance
selector = SelectFromModel(model, threshold="median", prefit=True)
selected_features = X.columns[selector.get_support()]
selected_features

Index(['systolic_bp_1', 'systolic_bp_2', 'systolic_bp_3', 'bmi',
       'neutrophil_count', 'basophil_perc', 'hemoglobin',
       'mean_cell_hemoglobin', 'mean_cell_volume', 'monocyte_perc',
       'neutrophil_perc', 'nucleated_red_blodd_cells',
       'red_cell_distribution_width', 'c_reactive_protein', 'albumin',
       'globulin_g_liter', 'albumin_g_dl', 'glucose_mg_dl',
       'ferritin_microgram', 'vitamin_d2', 'vitamin_d3', 'epi_vitamin_d3',
       'vitamin_d2_d3', 'minutes_vigorous_activity', 'minutes_sport_moderate',
       'minutes_sedentary', 'days_cigarette_smoked', 'cigarette_count',
       'days_pipe_smoked', 'days_chewing_tobacco', 'days_hookah',
       'hours_slept_weekday', 'hours_slept_weekend', 'recent_cocaine_use',
       'recent_alcohol_use'],
      dtype='object')

### Feature Importance via Performance Permutation 

This works by telling us much our model's performance decreases when a single feature’s values are randomly shuffled. For example, if we randomly shuffle the values in 'days_cigarette_smoke' row-wise and accuraacy significanty drops, then that feature is important.

In [24]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

importance_df.head(20)

,feature,importance
57,days_cigar_smoked,0.001183
5,diastolic_bp_2,0.000592
50,minutes_walking_bicycling,0.000394
65,recent_cocaine_use,0.000394
54,days_cigarette_smoked,0.000197
60,days_nicotine_replacement,0.000000
61,days_hookah,0.000000
67,recent_meth_use,0.000000
3,heart_rate,0.000000
66,recent_heroin_use,0.000000


### Splitting

We conduct a 60/20/20 split, stratified to ensure class distributions are preserved in each.

In [25]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

### Baseline Model

We use majority class prediction as our baseline

In [26]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Baseline Macro F1:", f1_score(y_test, y_pred_dummy, average="macro"))
print("Baseline Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_dummy))

Baseline Accuracy: 0.7445759368836292
Baseline Macro F1: 0.1707179197286603
Baseline Balanced Accuracy: 0.2


### Model 1: Random Forest

In [27]:
rf = RandomForestClassifier(
    n_estimators=400,
    class_weight=weight_dict, # to combat imbalances
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print("RF Macro F1:", f1_score(y_test, y_pred_rf, average="macro"))
print("RF Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_rf))

RF Accuracy: 0.7445759368836292
RF Macro F1: 0.1707179197286603
RF Balanced Accuracy: 0.2


In [28]:
confusion_matrix(y_test, y_pred_rf)

array([[755,   0,   0,   0,   0],
       [167,   0,   0,   0,   0],
       [ 59,   0,   0,   0,   0],
       [ 25,   0,   0,   0,   0],
       [  8,   0,   0,   0,   0]])

### Model 2: XGBoost

In [29]:
xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("XGB Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGB Macro F1:", f1_score(y_test, y_pred_xgb, average="macro"))
print("XGB Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_xgb))

XGB Accuracy: 0.7396449704142012
XGB Macro F1: 0.1889043216702791
XGB Balanced Accuracy: 0.20706983384224928


In [30]:
confusion_matrix(y_test, y_pred_xgb)

array([[741,  14,   0,   0,   0],
       [157,   9,   0,   1,   0],
       [ 55,   4,   0,   0,   0],
       [ 24,   1,   0,   0,   0],
       [  7,   0,   1,   0,   0]])

### Model Evaluation & Fairness